In [1]:
import pandas as pd

# Loading DataSet

In [2]:
df = pd.read_excel('Product-Sales-Region.xlsx')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           1500 non-null   datetime64[us]
 1   Region         1500 non-null   str           
 2   Product        1500 non-null   str           
 3   Quantity       1500 non-null   int64         
 4   UnitPrice      1500 non-null   float64       
 5   StoreLocation  1500 non-null   str           
 6   CustomerType   1500 non-null   str           
 7   Discount       1500 non-null   float64       
 8   Salesperson    1500 non-null   str           
 9   TotalPrice     1500 non-null   float64       
 10  PaymentMethod  1500 non-null   str           
 11  Promotion      1130 non-null   str           
 12  Returned       1500 non-null   int64         
 13  OrderID        1500 non-null   str           
 14  CustomerName   1500 non-null   str           
 15  ShippingCost   1500 non-null   f

In [4]:
df.shape

(1500, 19)

# Checking for Error

In [5]:
df.isna().sum()

Date               0
Region             0
Product            0
Quantity           0
UnitPrice          0
StoreLocation      0
CustomerType       0
Discount           0
Salesperson        0
TotalPrice         0
PaymentMethod      0
Promotion        370
Returned           0
OrderID            0
CustomerName       0
ShippingCost       0
OrderDate          0
DeliveryDate       0
RegionManager      0
dtype: int64

In [6]:
percentage = (df['Promotion'].isna().sum() / 1500) * 100
percentage

np.float64(24.666666666666668)

In [7]:
df.describe()

,Date,Quantity,UnitPrice,Discount,TotalPrice,Returned,ShippingCost,OrderDate,DeliveryDate
count,1500,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500,1500
mean,2024-04-07 21:35:02.400000,10.410667,298.826947,0.073133,2919.994952,0.248000,27.507293,2024-04-07 21:35:02.400000,2024-04-13 22:36:28.800000
min,2023-01-01 00:00:00,1.000000,5.520000,0.000000,6.970000,0.000000,5.010000,2023-01-01 00:00:00,2023-01-08 00:00:00
25%,2023-08-20 00:00:00,5.000000,151.020000,0.000000,867.136250,0.000000,16.700000,2023-08-20 00:00:00,2023-08-25 12:00:00
50%,2024-04-08 12:00:00,11.000000,294.740000,0.050000,2174.724000,0.000000,27.100000,2024-04-08 12:00:00,2024-04-15 12:00:00
75%,2024-12-07 06:00:00,15.000000,446.702500,0.100000,4414.723500,0.000000,38.635000,2024-12-07 06:00:00,2024-12-13 06:00:00
max,2025-06-30 00:00:00,20.000000,599.720000,0.150000,11077.000000,1.000000,49.980000,2025-06-30 00:00:00,2025-07-10 00:00:00
std,NaN,5.735732,169.100075,0.055979,2522.789977,0.431996,13.093453,NaN,NaN


# Handling Outliners

In [8]:
target_cols = ['TotalPrice','ShippingCost','UnitPrice','Quantity']
import numpy as np
def neutralize_cols(df,col_name):
    q1 = df[col_name].quantile(0.25)
    q3 = df[col_name].quantile(0.75)
    IQR = q3 - q1
    lower_bound = q1 - 1.5 * IQR
    upper_bound = q3 + 1.5 * IQR
    if col_name in ['ShippingCost','UnitPrice','Quantity']:
        lower_bound = max(0,lower_bound)
    df[col_name] = np.clip(df[col_name],lower_bound,upper_bound)
for col in target_cols:
    neutralize_cols(df,col)

In [9]:
df[target_cols].describe()

,TotalPrice,ShippingCost,UnitPrice,Quantity
count,1500.000000,1500.000000,1500.000000,1500.000000
mean,2913.362875,27.507293,298.826947,10.410667
std,2503.685969,13.093453,169.100075,5.735732
min,6.970000,5.010000,5.520000,1.000000
25%,867.136250,16.700000,151.020000,5.000000
50%,2174.724000,27.100000,294.740000,11.000000
75%,4414.723500,38.635000,446.702500,15.000000
max,9736.104375,49.980000,599.720000,20.000000


In [10]:
categorial_cols = df.select_dtypes(include=['str','category']).columns.tolist()
categorial_cols

['Region',
 'Product',
 'StoreLocation',
 'CustomerType',
 'Salesperson',
 'PaymentMethod',
 'Promotion',
 'OrderID',
 'CustomerName',
 'RegionManager']

# One Hot Encoding
#### Converting categorial values into numerical values to impute values for promotion columns using KNN 

In [11]:
df.nunique()

Date              747
Region              5
Product             7
Quantity           20
UnitPrice        1482
StoreLocation       4
CustomerType        2
Discount            4
Salesperson         6
TotalPrice       1484
PaymentMethod       5
Promotion           3
Returned            2
OrderID          1500
CustomerName     1371
ShippingCost     1285
OrderDate         747
DeliveryDate      733
RegionManager       5
dtype: int64

In [12]:
nominal_cols = ['Region','Product','StoreLocation','CustomerType','PaymentMethod']
df_encoded = pd.get_dummies(df,columns = nominal_cols,dtype=int)
df_encoded.head()

,Date,Quantity,UnitPrice,Discount,Salesperson,TotalPrice,Promotion,Returned,OrderID,CustomerName,...,StoreLocation_Store B,StoreLocation_Store C,StoreLocation_Store D,CustomerType_Retail,CustomerType_Wholesale,PaymentMethod_Cash,PaymentMethod_Credit Card,PaymentMethod_Debit Card,PaymentMethod_Gift Card,PaymentMethod_Online
0,2023-02-23,14,163.60,0.00,Eva,2290.400,FREESHIP,0,REG100000,Cust 6583,...,1,0,0,0,1,0,0,0,0,1
1,2024-12-19,1,544.01,0.00,Alice,544.010,SAVE10,0,REG100001,Cust 2144,...,0,0,0,1,0,0,0,0,1,0
2,2023-05-10,14,346.18,0.10,Alice,4361.868,WINTER15,0,REG100002,Cust 5998,...,1,0,0,0,1,0,0,0,0,1
3,2025-02-26,18,384.82,0.15,Frank,5887.746,FREESHIP,0,REG100003,Cust 7136,...,0,0,0,0,1,0,0,0,1,0
4,2023-06-24,18,237.76,0.00,Carlos,4279.680,SAVE10,0,REG100004,Cust 6506,...,0,1,0,1,0,0,0,0,0,1


## Handling missing values 
#### Using KNNImputer to compute the missing values

In [13]:
unique_promotion = df_encoded['Promotion'].dropna().unique()
unique_promotion

<StringArray>
['FREESHIP', 'SAVE10', 'WINTER15']
Length: 3, dtype: str

In [14]:
promo_mapping = {promo: idx for idx,promo in enumerate(unique_promotion)}
df_encoded['Promotion_Numeric'] = df_encoded['Promotion'].map(promo_mapping)
df_encoded.shape

(1500, 38)

In [15]:
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors = 5)

In [16]:
df_cols = df_encoded.drop(columns=['Salesperson','CustomerName','OrderID','RegionManager','Date','OrderDate','DeliveryDate','Promotion'])
df_cols.shape
df_cols['Promotion_Numeric'].isna().sum()

np.int64(370)

In [17]:
imputed_array = imputer.fit_transform(df_cols)
imputed_array.shape

(1500, 30)

In [18]:
imputer_df = pd.DataFrame(imputed_array,columns=df_cols.columns)

In [19]:
imputer_df['Promotion_Numeric'].isna().sum()

np.int64(0)

In [20]:
promo_mapping.items()

dict_items([('FREESHIP', 0), ('SAVE10', 1), ('WINTER15', 2)])

In [21]:
inverse_promo_mapping = {idx : promo for promo,idx in promo_mapping.items()}

In [22]:
inverse_promo_mapping

{0: 'FREESHIP', 1: 'SAVE10', 2: 'WINTER15'}

In [23]:
imputer_df['Promotion_Numeric'] = imputer_df['Promotion_Numeric'].round().astype(int)

In [24]:
df_encoded['Promotion'] = imputer_df['Promotion_Numeric'].map(inverse_promo_mapping)

In [25]:
df_encoded.isna().sum()

Date                           0
Quantity                       0
UnitPrice                      0
Discount                       0
Salesperson                    0
TotalPrice                     0
Promotion                      0
Returned                       0
OrderID                        0
CustomerName                   0
ShippingCost                   0
OrderDate                      0
DeliveryDate                   0
RegionManager                  0
Region_Central                 0
Region_East                    0
Region_North                   0
Region_South                   0
Region_West                    0
Product_Chair                  0
Product_Desk                   0
Product_Laptop                 0
Product_Monitor                0
Product_Phone                  0
Product_Printer                0
Product_Tablet                 0
StoreLocation_Store A          0
StoreLocation_Store B          0
StoreLocation_Store C          0
StoreLocation_Store D          0
CustomerTy

In [26]:
df_encoded.drop(columns='Promotion_Numeric')

,Date,Quantity,UnitPrice,Discount,Salesperson,TotalPrice,Promotion,Returned,OrderID,CustomerName,...,StoreLocation_Store B,StoreLocation_Store C,StoreLocation_Store D,CustomerType_Retail,CustomerType_Wholesale,PaymentMethod_Cash,PaymentMethod_Credit Card,PaymentMethod_Debit Card,PaymentMethod_Gift Card,PaymentMethod_Online
0,2023-02-23,14,163.60,0.00,Eva,2290.400,FREESHIP,0,REG100000,Cust 6583,...,1,0,0,0,1,0,0,0,0,1
1,2024-12-19,1,544.01,0.00,Alice,544.010,SAVE10,0,REG100001,Cust 2144,...,0,0,0,1,0,0,0,0,1,0
2,2023-05-10,14,346.18,0.10,Alice,4361.868,WINTER15,0,REG100002,Cust 5998,...,1,0,0,0,1,0,0,0,0,1
3,2025-02-26,18,384.82,0.15,Frank,5887.746,FREESHIP,0,REG100003,Cust 7136,...,0,0,0,0,1,0,0,0,1,0
4,2023-06-24,18,237.76,0.00,Carlos,4279.680,SAVE10,0,REG100004,Cust 6506,...,0,1,0,1,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,2025-02-17,13,134.56,0.05,Carlos,1661.816,WINTER15,0,REG101495,Cust 5227,...,0,0,1,1,0,0,1,0,0,0
1496,2024-01-11,18,209.75,0.15,Frank,3209.175,FREESHIP,0,REG101496,Cust 5559,...,0,1,0,0,1,0,0,1,0,0
1497,2024-07-27,1,272.50,0.00,Eva,272.500,WINTER15,0,REG101497,Cust 8981,...,0,0,0,0,1,0,0,0,1,0
1498,2024-12-03,14,262.67,0.05,Carlos,3493.511,SAVE10,0,REG101498,Cust 1824,...,0,0,0,1,0,0,0,0,1,0


# Feature Engineering

In [27]:
df_encoded['DeliveryTime'] = (pd.to_datetime(df_encoded['DeliveryDate']) - pd.to_datetime(df_encoded['OrderDate'])).dt.days
df['ShippingCost_Ratio'] = df['ShippingCost'] / df['TotalPrice']
df_encoded['Avg_Price_Per_Item'] = df_encoded['TotalPrice'] / df_encoded['Quantity']
print("Successfully engineered 3 new predictive features!")

Successfully engineered 3 new predictive features!


In [28]:
df_encoded.head(10)

,Date,Quantity,UnitPrice,Discount,Salesperson,TotalPrice,Promotion,Returned,OrderID,CustomerName,...,CustomerType_Retail,CustomerType_Wholesale,PaymentMethod_Cash,PaymentMethod_Credit Card,PaymentMethod_Debit Card,PaymentMethod_Gift Card,PaymentMethod_Online,Promotion_Numeric,DeliveryTime,Avg_Price_Per_Item
0,2023-02-23,14,163.60,0.00,Eva,2290.400,FREESHIP,0,REG100000,Cust 6583,...,0,1,0,0,0,0,1,0.0,4,163.6000
1,2024-12-19,1,544.01,0.00,Alice,544.010,SAVE10,0,REG100001,Cust 2144,...,1,0,0,0,0,1,0,1.0,9,544.0100
2,2023-05-10,14,346.18,0.10,Alice,4361.868,WINTER15,0,REG100002,Cust 5998,...,0,1,0,0,0,0,1,2.0,9,311.5620
3,2025-02-26,18,384.82,0.15,Frank,5887.746,FREESHIP,0,REG100003,Cust 7136,...,0,1,0,0,0,1,0,0.0,4,327.0970
4,2023-06-24,18,237.76,0.00,Carlos,4279.680,SAVE10,0,REG100004,Cust 6506,...,1,0,0,0,0,0,1,1.0,3,237.7600
5,2024-02-20,2,385.09,0.15,Diana,654.653,FREESHIP,0,REG100005,Cust 3909,...,0,1,0,1,0,0,0,NaN,10,327.3265
6,2023-01-11,7,17.50,0.10,Alice,110.250,WINTER15,0,REG100006,Cust 7887,...,1,0,0,0,1,0,0,2.0,3,15.7500
7,2023-01-09,3,330.22,0.15,Carlos,842.061,SAVE10,0,REG100007,Cust 5301,...,0,1,1,0,0,0,0,NaN,2,280.6870
8,2024-10-16,3,432.04,0.05,Bob,1231.314,FREESHIP,1,REG100008,Cust 2284,...,0,1,0,0,1,0,0,0.0,2,410.4380
9,2025-03-05,5,323.28,0.10,Frank,1454.760,SAVE10,0,REG100009,Cust 3732,...,1,0,0,0,0,1,0,1.0,8,290.9520


In [29]:
df_encoded = df_encoded.drop(columns='Promotion_Numeric')

In [30]:
df_encoded.head(10)

,Date,Quantity,UnitPrice,Discount,Salesperson,TotalPrice,Promotion,Returned,OrderID,CustomerName,...,StoreLocation_Store D,CustomerType_Retail,CustomerType_Wholesale,PaymentMethod_Cash,PaymentMethod_Credit Card,PaymentMethod_Debit Card,PaymentMethod_Gift Card,PaymentMethod_Online,DeliveryTime,Avg_Price_Per_Item
0,2023-02-23,14,163.60,0.00,Eva,2290.400,FREESHIP,0,REG100000,Cust 6583,...,0,0,1,0,0,0,0,1,4,163.6000
1,2024-12-19,1,544.01,0.00,Alice,544.010,SAVE10,0,REG100001,Cust 2144,...,0,1,0,0,0,0,1,0,9,544.0100
2,2023-05-10,14,346.18,0.10,Alice,4361.868,WINTER15,0,REG100002,Cust 5998,...,0,0,1,0,0,0,0,1,9,311.5620
3,2025-02-26,18,384.82,0.15,Frank,5887.746,FREESHIP,0,REG100003,Cust 7136,...,0,0,1,0,0,0,1,0,4,327.0970
4,2023-06-24,18,237.76,0.00,Carlos,4279.680,SAVE10,0,REG100004,Cust 6506,...,0,1,0,0,0,0,0,1,3,237.7600
5,2024-02-20,2,385.09,0.15,Diana,654.653,FREESHIP,0,REG100005,Cust 3909,...,1,0,1,0,1,0,0,0,10,327.3265
6,2023-01-11,7,17.50,0.10,Alice,110.250,WINTER15,0,REG100006,Cust 7887,...,0,1,0,0,0,1,0,0,3,15.7500
7,2023-01-09,3,330.22,0.15,Carlos,842.061,SAVE10,0,REG100007,Cust 5301,...,0,0,1,1,0,0,0,0,2,280.6870
8,2024-10-16,3,432.04,0.05,Bob,1231.314,FREESHIP,1,REG100008,Cust 2284,...,1,0,1,0,0,1,0,0,2,410.4380
9,2025-03-05,5,323.28,0.10,Frank,1454.760,SAVE10,0,REG100009,Cust 3732,...,1,1,0,0,0,0,1,0,8,290.9520


In [32]:
df_encoded.columns

Index(['Date', 'Quantity', 'UnitPrice', 'Discount', 'Salesperson',
       'TotalPrice', 'Promotion', 'Returned', 'OrderID', 'CustomerName',
       'ShippingCost', 'OrderDate', 'DeliveryDate', 'RegionManager',
       'Region_Central', 'Region_East', 'Region_North', 'Region_South',
       'Region_West', 'Product_Chair', 'Product_Desk', 'Product_Laptop',
       'Product_Monitor', 'Product_Phone', 'Product_Printer', 'Product_Tablet',
       'StoreLocation_Store A', 'StoreLocation_Store B',
       'StoreLocation_Store C', 'StoreLocation_Store D', 'CustomerType_Retail',
       'CustomerType_Wholesale', 'PaymentMethod_Cash',
       'PaymentMethod_Credit Card', 'PaymentMethod_Debit Card',
       'PaymentMethod_Gift Card', 'PaymentMethod_Online', 'DeliveryTime',
       'Avg_Price_Per_Item'],
      dtype='str')

## Eradicating MultiCollinearity

In [35]:
numeric_df = df_encoded.select_dtypes(include = [np.number])
numeric_df

,Quantity,UnitPrice,Discount,TotalPrice,Returned,ShippingCost,Region_Central,Region_East,Region_North,Region_South,...,StoreLocation_Store D,CustomerType_Retail,CustomerType_Wholesale,PaymentMethod_Cash,PaymentMethod_Credit Card,PaymentMethod_Debit Card,PaymentMethod_Gift Card,PaymentMethod_Online,DeliveryTime,Avg_Price_Per_Item
0,14,163.60,0.00,2290.400,0,43.34,0,1,0,0,...,0,0,1,0,0,0,0,1,4,163.6000
1,1,544.01,0.00,544.010,0,5.30,0,0,0,1,...,0,1,0,0,0,0,1,0,9,544.0100
2,14,346.18,0.10,4361.868,0,20.46,0,0,1,0,...,0,0,1,0,0,0,0,1,9,311.5620
3,18,384.82,0.15,5887.746,0,27.95,1,0,0,0,...,0,0,1,0,0,0,1,0,4,327.0970
4,18,237.76,0.00,4279.680,0,5.73,0,1,0,0,...,0,1,0,0,0,0,0,1,3,237.7600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,13,134.56,0.05,1661.816,0,35.63,0,0,0,0,...,1,1,0,0,1,0,0,0,5,127.8320
1496,18,209.75,0.15,3209.175,0,45.93,0,0,0,1,...,0,0,1,0,0,1,0,0,6,178.2875
1497,1,272.50,0.00,272.500,0,35.56,0,1,0,0,...,0,0,1,0,0,0,1,0,7,272.5000
1498,14,262.67,0.05,3493.511,0,24.53,1,0,0,0,...,0,1,0,0,0,0,1,0,7,249.5365


In [36]:
corr_matrix = numeric_df.corr().abs()

In [37]:
corr_matrix

,Quantity,UnitPrice,Discount,TotalPrice,Returned,ShippingCost,Region_Central,Region_East,Region_North,Region_South,...,StoreLocation_Store D,CustomerType_Retail,CustomerType_Wholesale,PaymentMethod_Cash,PaymentMethod_Credit Card,PaymentMethod_Debit Card,PaymentMethod_Gift Card,PaymentMethod_Online,DeliveryTime,Avg_Price_Per_Item
Quantity,1.000000,0.042239,0.017660,0.667410,0.031832,0.017973,0.009177,0.045808,0.064715,0.034560,...,0.006643,0.015240,0.015240,0.018870,0.009543,0.011804,0.016006,0.017072,0.013434,0.037850
UnitPrice,0.042239,1.000000,0.006790,0.678141,0.020117,0.003202,0.007899,0.020332,0.008045,0.009359,...,0.015249,0.031725,0.031725,0.032067,0.007914,0.080593,0.045981,0.010111,0.010575,0.992621
Discount,0.017660,0.006790,1.000000,0.057713,0.006742,0.030558,0.010020,0.016000,0.022146,0.000769,...,0.026850,0.055727,0.055727,0.052317,0.027341,0.007280,0.039501,0.005157,0.008122,0.097063
TotalPrice,0.667410,0.678141,0.057713,1.000000,0.015041,0.009728,0.021209,0.016231,0.044461,0.022464,...,0.021660,0.012429,0.012429,0.025318,0.004145,0.050755,0.011936,0.016539,0.017735,0.678534
Returned,0.031832,0.020117,0.006742,0.015041,1.000000,0.008132,0.017916,0.014744,0.025315,0.014914,...,0.002793,0.013857,0.013857,0.070836,0.035365,0.023767,0.049593,0.011657,0.001902,0.020074
ShippingCost,0.017973,0.003202,0.030558,0.009728,0.008132,1.000000,0.014608,0.030221,0.062560,0.035479,...,0.007927,0.013355,0.013355,0.001297,0.006684,0.000261,0.021447,0.012691,0.027636,0.006247
Region_Central,0.009177,0.007899,0.010020,0.021209,0.017916,0.014608,1.000000,0.256249,0.255210,0.247908,...,0.017817,0.032307,0.032307,0.018898,0.008331,0.009245,0.024126,0.012897,0.032685,0.007541
Region_East,0.045808,0.020332,0.016000,0.016231,0.014744,0.030221,0.256249,1.000000,0.260503,0.253050,...,0.010908,0.023865,0.023865,0.005316,0.008203,0.000527,0.033001,0.027879,0.009244,0.022581
Region_North,0.064715,0.008045,0.022146,0.044461,0.025315,0.062560,0.255210,0.260503,1.000000,0.252023,...,0.012674,0.045287,0.045287,0.043570,0.006577,0.065701,0.017629,0.009873,0.068716,0.011637
Region_South,0.034560,0.009359,0.000769,0.022464,0.014914,0.035479,0.247908,0.253050,0.252023,1.000000,...,0.025285,0.021201,0.021201,0.031571,0.009222,0.021094,0.044497,0.018457,0.021531,0.009101


In [40]:
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape),k=1).astype(bool))
upper_triangle

,Quantity,UnitPrice,Discount,TotalPrice,Returned,ShippingCost,Region_Central,Region_East,Region_North,Region_South,...,StoreLocation_Store D,CustomerType_Retail,CustomerType_Wholesale,PaymentMethod_Cash,PaymentMethod_Credit Card,PaymentMethod_Debit Card,PaymentMethod_Gift Card,PaymentMethod_Online,DeliveryTime,Avg_Price_Per_Item
Quantity,NaN,0.042239,0.01766,0.667410,0.031832,0.017973,0.009177,0.045808,0.064715,0.034560,...,0.006643,0.015240,0.015240,0.018870,0.009543,0.011804,0.016006,0.017072,0.013434,0.037850
UnitPrice,NaN,NaN,0.00679,0.678141,0.020117,0.003202,0.007899,0.020332,0.008045,0.009359,...,0.015249,0.031725,0.031725,0.032067,0.007914,0.080593,0.045981,0.010111,0.010575,0.992621
Discount,NaN,NaN,NaN,0.057713,0.006742,0.030558,0.010020,0.016000,0.022146,0.000769,...,0.026850,0.055727,0.055727,0.052317,0.027341,0.007280,0.039501,0.005157,0.008122,0.097063
TotalPrice,NaN,NaN,NaN,NaN,0.015041,0.009728,0.021209,0.016231,0.044461,0.022464,...,0.021660,0.012429,0.012429,0.025318,0.004145,0.050755,0.011936,0.016539,0.017735,0.678534
Returned,NaN,NaN,NaN,NaN,NaN,0.008132,0.017916,0.014744,0.025315,0.014914,...,0.002793,0.013857,0.013857,0.070836,0.035365,0.023767,0.049593,0.011657,0.001902,0.020074
ShippingCost,NaN,NaN,NaN,NaN,NaN,NaN,0.014608,0.030221,0.062560,0.035479,...,0.007927,0.013355,0.013355,0.001297,0.006684,0.000261,0.021447,0.012691,0.027636,0.006247
Region_Central,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.256249,0.255210,0.247908,...,0.017817,0.032307,0.032307,0.018898,0.008331,0.009245,0.024126,0.012897,0.032685,0.007541
Region_East,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.260503,0.253050,...,0.010908,0.023865,0.023865,0.005316,0.008203,0.000527,0.033001,0.027879,0.009244,0.022581
Region_North,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.252023,...,0.012674,0.045287,0.045287,0.043570,0.006577,0.065701,0.017629,0.009873,0.068716,0.011637
Region_South,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.025285,0.021201,0.021201,0.031571,0.009222,0.021094,0.044497,0.018457,0.021531,0.009101


In [44]:
collinear_pairs = [(column,row) for column in upper_triangle.columns for row in upper_triangle.index if upper_triangle[column][row] > 0.80]

In [45]:
collinear_pairs

[('CustomerType_Wholesale', 'CustomerType_Retail'),
 ('Avg_Price_Per_Item', 'UnitPrice')]

In [46]:
target_variable = 'TotalPrice'

In [47]:
columns_todrop = set()

In [48]:
for col1,col2 in collinear_pairs:
    if col1 == target_variable or col2 == target_variable:
        continue
    corr_col1 = numeric_df[col1].corr(numeric_df[target_variable])
    corr_col2 = numeric_df[col2].corr(numeric_df[target_variable])
    if abs(corr_col1) < abs(corr_col2):
        columns_todrop.add(col1)
    else:
        columns_todrop.add(col2)

In [50]:
columns_todr

{'CustomerType_Retail', 'UnitPrice'}

In [53]:
df_final = df_encoded.drop(columns = list(columns_todrop))
print("Dropped high collinearity columns:",list(columns_todrop))

Dropped high collinearity columns: ['CustomerType_Retail', 'UnitPrice']


In [54]:
df_final.head(10)

,Date,Quantity,Discount,Salesperson,TotalPrice,Promotion,Returned,OrderID,CustomerName,ShippingCost,...,StoreLocation_Store C,StoreLocation_Store D,CustomerType_Wholesale,PaymentMethod_Cash,PaymentMethod_Credit Card,PaymentMethod_Debit Card,PaymentMethod_Gift Card,PaymentMethod_Online,DeliveryTime,Avg_Price_Per_Item
0,2023-02-23,14,0.00,Eva,2290.400,FREESHIP,0,REG100000,Cust 6583,43.34,...,0,0,1,0,0,0,0,1,4,163.6000
1,2024-12-19,1,0.00,Alice,544.010,SAVE10,0,REG100001,Cust 2144,5.30,...,0,0,0,0,0,0,1,0,9,544.0100
2,2023-05-10,14,0.10,Alice,4361.868,WINTER15,0,REG100002,Cust 5998,20.46,...,0,0,1,0,0,0,0,1,9,311.5620
3,2025-02-26,18,0.15,Frank,5887.746,FREESHIP,0,REG100003,Cust 7136,27.95,...,0,0,1,0,0,0,1,0,4,327.0970
4,2023-06-24,18,0.00,Carlos,4279.680,SAVE10,0,REG100004,Cust 6506,5.73,...,1,0,0,0,0,0,0,1,3,237.7600
5,2024-02-20,2,0.15,Diana,654.653,FREESHIP,0,REG100005,Cust 3909,11.92,...,0,1,1,0,1,0,0,0,10,327.3265
6,2023-01-11,7,0.10,Alice,110.250,WINTER15,0,REG100006,Cust 7887,5.02,...,1,0,0,0,0,1,0,0,3,15.7500
7,2023-01-09,3,0.15,Carlos,842.061,SAVE10,0,REG100007,Cust 5301,48.01,...,1,0,1,1,0,0,0,0,2,280.6870
8,2024-10-16,3,0.05,Bob,1231.314,FREESHIP,1,REG100008,Cust 2284,42.11,...,0,1,1,0,0,1,0,0,2,410.4380
9,2025-03-05,5,0.10,Frank,1454.760,SAVE10,0,REG100009,Cust 3732,41.10,...,0,1,0,0,0,0,1,0,8,290.9520


## Output

In [55]:
df_final.to_csv("Product-Region-Sales.csv",index = False)
print("DataSet is Successfully preprocessed for ML Analysis.")

DataSet is Successfully preprocessed for ML Analysis.
